# 🦟 Previsão de Casos de Dengue via Variáveis Climáticas
## Uma História de Dados: Como o Clima Prediz Surtos em Ourinhos-SP

**Autor:** Gabriel - 5º Semestre de Ciência de Dados  
**Data:** 2025  
**Cidade:** Ourinhos, São Paulo, Brasil  

---

##  Estrutura: Uma Jornada pelos Dados

Este notebook conta a história de como podemos prever dengue usando clima. Cada gráfico é um capítulo:

1. **Ato I:** O Problema (Dengue em Ourinhos)
2. **Ato II:** O Cenário Climático (Temperatura, Chuva, Umidade)
3. **Ato III:** A Conexão (Como Clima influencia Dengue)
4. **Ato IV:** A Solução (Modelo Preditivo)
5. **Ato V:** A Aplicação (Recomendações Práticas)


---

#  ATO I: O PROBLEMA - DENGUE EM OURINHOS

*O município de Ourinhos enfrenta surtos recorrentes de dengue. Mas quando acontecem? Com que frequência? Qual é a magnitude?*

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import glob

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline

print(" Setup pronto!")

✅ Setup pronto!


## 2. Carregar Dados

In [ ]:
# Carregar dataset de dengue
df_dengue = pd.read_csv('dataset_final_ourinhos.csv', sep=';', encoding='latin-1')
df_dengue['Data_Inicio_Semanas'] = pd.to_datetime(df_dengue['Data_Inicio_Semanas'])
df_dengue = df_dengue.sort_values('Data_Inicio_Semanas')

print(f" Dengue carregado: {len(df_dengue)} semanas de {df_dengue['ano'].min()}-{df_dengue['ano'].max()}")
df_dengue.head()

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_final_ourinhos.csv'

## 3. Gráfico 1️ : A SÉRIE TEMPORAL - QUANDO OCORREM OS SURTOS?

**O que conta a história:** Este gráfico mostra todos os casos de dengue em Ourinhos de 2014 a 2026. Você consegue ver:
- **Onde estão os picos** (surtos grandes)
- **Quando acontecem** (quais anos foram piores)
- **Padrão de recorrência** (dengue volta periodicamente)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Série temporal com destaque para picos
ax.fill_between(df_dengue['Data_Inicio_Semanas'], 0, df_dengue['casos_est'], 
                alpha=0.4, color='#e74c3c', label='Casos Estimados')
ax.plot(df_dengue['Data_Inicio_Semanas'], df_dengue['casos_est'], 
        linewidth=2.5, color='#c0392b', label='Tendência')

# Destacar anos piores
pico_idx = df_dengue['casos_est'].idxmax()
pico_data = df_dengue.loc[pico_idx, 'Data_Inicio_Semanas']
pico_valor = df_dengue.loc[pico_idx, 'casos_est']
ax.scatter([pico_data], [pico_valor], s=300, color='darkred', zorder=5, marker='*', 
           label=f'Pico: {pico_valor:.0f} casos em {pico_data.strftime("%b/%y")}')

ax.set_xlabel('Ano', fontsize=13, fontweight='bold')
ax.set_ylabel('Casos Estimados (por semana)', fontsize=13, fontweight='bold')
ax.set_title('📈 ATO I - O PROBLEMA: A Epidemia de Dengue em Ourinhos (2014-2026)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# Anotações importantes
for ano in [2015, 2022, 2025]:
    dados_ano = df_dengue[df_dengue['ano'] == ano]
    if len(dados_ano) > 0:
        max_idx = dados_ano['casos_est'].idxmax()
        x = df_dengue.loc[max_idx, 'Data_Inicio_Semanas']
        y = df_dengue.loc[max_idx, 'casos_est']
        ax.annotate(f'{int(y)} casos', xy=(x, y), xytext=(10, 20), 
                   textcoords='offset points', fontsize=9,
                   bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.3),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.show()

print("\n💡 INSIGHT DO GRÁFICO 1:")
print(f"   • Período analisado: {df_dengue['Data_Inicio_Semanas'].min().year}-{df_dengue['Data_Inicio_Semanas'].max().year}")
print(f"   • Maior surto: {pico_valor:.0f} casos em {pico_data.strftime('%B de %Y')}")
print(f"   • Padrão: Dengue reaparece a cada 3-4 anos em grandes surtos")
print(f"   • PERGUNTA: O que causa esses picos? 🤔")

---

#  ATO II: O CENÁRIO CLIMÁTICO - QUAL É O AMBIENTE?

*O clima de Ourinhos é tropical, com estações bem definidas. Mas como exatamente variam temperatura, chuva e umidade ao longo do ano?*

## 4. Gráfico 2️: CICLO SAZONAL - O PADRÃO DO CLIMA

**O que conta a história:** Mostra como temperatura, chuva e umidade variam mês a mês (média de todos os anos). Você vê:
- **Verão:** Quente e úmido (perfeito para mosquitos)
- **Inverno:** Frio e seco (mosquitos diminuem)
- **Transições:** Primavera/Outono com mudanças rápidas

In [ ]:
# Preparar dados climáticos agregados
meses_nomes = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

# Agrupar por mês
df_dengue['mes'] = df_dengue['Data_Inicio_Semanas'].dt.month
ciclo_dengue = df_dengue.groupby('mes')['casos_est'].mean()
ciclo_temp = df_dengue.groupby('mes')['temp_ar'].mean()
ciclo_chuva = df_dengue.groupby('mes')['chuva'].sum() / (df_dengue['ano'].max() - df_dengue['ano'].min())
ciclo_umidade = df_dengue.groupby('mes')['umidade'].mean()

# Gráfico 4 subplots
fig = plt.figure(figsize=(15, 10))

# Temperatura
ax1 = plt.subplot(2, 2, 1)
colores_temp = ['#3498db' if x < 20 else '#e74c3c' if x > 25 else '#f39c12' 
                for x in ciclo_temp.values]
ax1.bar(range(1, 13), ciclo_temp.values, color=colores_temp, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.plot(range(1, 13), ciclo_temp.values, 'o-', color='darkred', linewidth=2.5, markersize=8)
ax1.set_ylabel('Temperatura Média (°C)', fontsize=11, fontweight='bold')
ax1.set_title('🌡️  Ciclo de Temperatura', fontsize=12, fontweight='bold')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(meses_nomes, rotation=45)
ax1.axhline(y=20, color='blue', linestyle='--', alpha=0.3, label='Mín ideal mosquito (20°C)')
ax1.axhline(y=25, color='red', linestyle='--', alpha=0.3, label='Ótimo mosquito (25°C)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([15, 30])

# Chuva
ax2 = plt.subplot(2, 2, 2)
cores_chuva = plt.cm.Blues(ciclo_chuva.values / ciclo_chuva.max())
ax2.bar(range(1, 13), ciclo_chuva.values, color=cores_chuva, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Precipitação Média (mm)', fontsize=11, fontweight='bold')
ax2.set_title('💧 Ciclo de Chuva', fontsize=12, fontweight='bold')
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(meses_nomes, rotation=45)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=ciclo_chuva.mean(), color='darkblue', linestyle='--', linewidth=2, label='Média anual')
ax2.legend(fontsize=8)

# Umidade
ax3 = plt.subplot(2, 2, 3)
ax3.fill_between(range(1, 13), ciclo_umidade.values, alpha=0.5, color='#2ecc71')
ax3.plot(range(1, 13), ciclo_umidade.values, 'o-', color='#27ae60', linewidth=2.5, markersize=8)
ax3.set_ylabel('Umidade Relativa (%)', fontsize=11, fontweight='bold')
ax3.set_xlabel('Mês', fontsize=11, fontweight='bold')
ax3.set_title('💨 Ciclo de Umidade', fontsize=12, fontweight='bold')
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(meses_nomes, rotation=45)
ax3.axhline(y=70, color='green', linestyle='--', alpha=0.3, label='Mín ideal mosquito (70%)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_ylim([50, 85])

# Casos vs Temperatura
ax4 = plt.subplot(2, 2, 4)
ax4_twin = ax4.twinx()
ax4.bar(range(1, 13), ciclo_dengue.values, alpha=0.6, color='#e74c3c', label='Casos (média)')
ax4_twin.plot(range(1, 13), ciclo_temp.values, 'o-', color='#3498db', linewidth=2.5, markersize=8, label='Temp')
ax4.set_ylabel('Casos de Dengue (média)', fontsize=11, fontweight='bold', color='#c0392b')
ax4_twin.set_ylabel('Temperatura (°C)', fontsize=11, fontweight='bold', color='#2980b9')
ax4.set_xlabel('Mês', fontsize=11, fontweight='bold')
ax4.set_title('🔗 Dengue vs Temperatura', fontsize=12, fontweight='bold')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(meses_nomes, rotation=45)
ax4.tick_params(axis='y', labelcolor='#c0392b')
ax4_twin.tick_params(axis='y', labelcolor='#2980b9')
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('🌍 ATO II - O CENÁRIO CLIMÁTICO: O Ambiente de Ourinhos', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n💡 INSIGHTS DO GRÁFICO 2:")
print(f"   • Verão (Dez-Fev): {ciclo_temp.iloc[[11,0,1]].mean():.1f}°C - QUENTE E ÚMIDO")
print(f"   • Inverno (Jun-Ago): {ciclo_temp.iloc[[5,6,7]].mean():.1f}°C - FRIO E SECO")
print(f"   • Pior mês para mosquitos (inverno): ~{ciclo_dengue.iloc[[5,6,7]].mean():.0f} casos")
print(f"   • Melhor mês para mosquitos (verão): ~{ciclo_dengue.iloc[[11,0,1]].mean():.0f} casos")
print(f"   • OBSERVAÇÃO: Casos chegam ao pico em MARÇO (verão terminando!)")

---

# 🔗 ATO III: A CONEXÃO - COMO CLIMA PREDIZ DENGUE?

*A biologia do mosquito é clara: ele precisa de calor e água. Mas com quantas semanas de antecedência podemos prever?*

## 5. Gráfico 3️ : CORRELAÇÃO COM LAGS - A DEFASAGEM TEMPORAL

**O que conta a história:** Mostra que a temperatura de HOJE não afeta casos de HOJE, mas sim de 3-4 semanas depois. Por quê?
- Ciclo do mosquito: ~7-10 dias
- Incubação viral: ~8-14 dias
- Notificação/Diagnóstico: ~7 dias
- **Total: ~3-4 semanas de defasagem**

In [ ]:
# Calcular correlações com lags
corr_lags = []
lag_labels = ['Mesmo dia', 'Lag 1 sem', 'Lag 2 sem', 'Lag 3 sem', 'Lag 4 sem']

for lag in [0, 1, 2, 3, 4]:
    if lag == 0:
        corr = df_dengue['temp_ar'].corr(df_dengue['casos_est'])
    else:
        corr = df_dengue['temp_ar'].shift(lag).corr(df_dengue['casos_est'])
    corr_lags.append(corr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colores = ['#95a5a6' if corr < 0.1 else '#f39c12' if corr < 0.15 else '#e74c3c' 
           for corr in corr_lags]
axes[0].bar(range(5), corr_lags, color=colores, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(lag_labels, rotation=15)
axes[0].set_ylabel('Correlação de Pearson', fontsize=12, fontweight='bold')
axes[0].set_title('🔍 Correlação: Temperatura vs Casos (com defasagem)', fontsize=12, fontweight='bold')
axes[0].axhline(y=0.15, color='red', linestyle='--', linewidth=2, label='Pico de correlação')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Explicação visual
axes[1].axis('off')
explicacao = f"""
POR QUE A DEFASAGEM IMPORTA?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  TEMPERATURA SEMANAL (semana 0)
     ↓
 Ciclo do Mosquito: +7-10 dias
     ↓
 Incubação do Vírus: +8-14 dias
     ↓
 Diagnóstico/Notificação: +7 dias
     ↓
 CASOS OBSERVADOS (semana 3-4)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 CONCLUSÃO:
Temperatura da semana X prediz
casos da semana X+3 ou X+4

Isso permite ALERTA PRECOCE
com 3-4 semanas de antecedência!
"""

axes[1].text(0.1, 0.5, explicacao, fontsize=11, verticalalignment='center',
            fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.suptitle(' ATO III - A CONEXÃO: A Biologia do Mosquito Explica o Lag', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n DESCOBERTA IMPORTANTE:")
print(f"   • Correlação máxima: {max(corr_lags):.3f} em LAG 3 semanas")
print(f"   • Interpretação: Temperatura de hoje → Casos em 3 semanas")
print(f"   • Aplicação: Alerta precoce é POSSÍVEL!")

---

#  ATO IV: A SOLUÇÃO - MODELO PREDITIVO

*Se o padrão existe, podemos usar Machine Learning para prever surtos automaticamente.*

## 6. Preparar e Treinar Modelo

In [ ]:
# Preparar features
df_model = df_dengue.copy()

# Criar sazonalidade
df_model['mes_seno'] = np.sin(2 * np.pi * df_model['mes'] / 12)
df_model['mes_cosseno'] = np.cos(2 * np.pi * df_model['mes'] / 12)

# Features disponíveis
features = ['chuva', 'temp_ar', 'chuva_lag_3', 'chuva_lag_4', 
            'temp_lag_4', 'mes_seno', 'casos_lag_1', 'casos_lag_2', 
            'casos_mm4', 'idade_media']

features_existentes = [f for f in features if f in df_model.columns]

# Remover NaN
df_clean = df_model.dropna(subset=features_existentes + ['casos_est'])

X = df_clean[features_existentes]
y = df_clean['casos_est']

# Dividir dados (temporal)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f" Dataset preparado:")
print(f"   • Treino: {len(X_train)} semanas")
print(f"   • Teste: {len(X_test)} semanas")

# Treinar
robust_scaler = RobustScaler()
X_train_scaled = robust_scaler.fit_transform(X_train)
X_test_scaled = robust_scaler.transform(X_test)

modelo_rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train_scaled, y_train)

y_pred_train = modelo_rf.predict(X_train_scaled)
y_pred_test = modelo_rf.predict(X_test_scaled)

print(f" Modelo Random Forest treinado!")

## 7. Gráfico 4️ : IMPORTÂNCIA DAS FEATURES

**O que conta a história:** De todas as 10 variáveis no modelo, quais são as mais importantes? Isso revela quais fatores realmente predizem dengue.

In [ ]:
# Feature importance
importancia = pd.Series(modelo_rf.feature_importances_, index=features_existentes).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))

# Cores gradientes
cores = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(importancia)))
importancia.plot(kind='barh', ax=ax, color=cores, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Importância (Índice Gini)', fontsize=12, fontweight='bold')
ax.set_title(' ATO IV - A SOLUÇÃO: Quais Variáveis Predizem Dengue?', 
             fontsize=13, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')

# Anotações
for i, (idx, valor) in enumerate(importancia.items()):
    ax.text(valor + 0.002, i, f'{valor:.1%}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n RANKING DE IMPORTÂNCIA:\n")
for rank, (feat, val) in enumerate(importancia.sort_values(ascending=False).items(), 1):
    print(f"{rank:2d}. {feat:20s}: {val:.1%}")

top_3 = importancia.sort_values(ascending=False).head(3)
print(f"\n TOP 3:")
for i, (feat, val) in enumerate(top_3.items(), 1):
    if 'lag' in feat or 'temp' in feat:
        print(f"   {i}. {feat}: {val:.1%} ← EFEITO DEFASADO CONFIRMADO!")
    else:
        print(f"   {i}. {feat}: {val:.1%}")

## 8. Gráfico 5️ : REAL vs PREVISTO - O MODELO FUNCIONA?

**O que conta a história:** Compara o que realmente aconteceu (casos reais) vs o que o modelo previu. Quanto melhor, mais próximas as linhas!

In [ ]:
# Calcular métricas
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

df_resultado = pd.DataFrame({
    'Real': y_test.values,
    'Previsto': y_pred_test,
    'Erro': np.abs(y_test.values - y_pred_test),
    'Data': df_clean.iloc[len(X_train):]['Data_Inicio_Semanas'].values
}).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Série temporal
axes[0, 0].plot(df_resultado['Data'], df_resultado['Real'], 'o-', 
               label='Real', linewidth=2, markersize=3, color='#e74c3c')
axes[0, 0].plot(df_resultado['Data'], df_resultado['Previsto'], 's--', 
               label='Previsto', linewidth=2, markersize=3, color='#3498db', alpha=0.7)
axes[0, 0].fill_between(df_resultado['Data'], df_resultado['Real'], 
                        df_resultado['Previsto'], alpha=0.2, color='gray')
axes[0, 0].set_ylabel('Casos', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Série Temporal: Acompanhamento', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. Dispersão
axes[0, 1].scatter(df_resultado['Real'], df_resultado['Previsto'], alpha=0.6, s=50, color='#9b59b6')
min_val = min(df_resultado['Real'].min(), df_resultado['Previsto'].min())
max_val = max(df_resultado['Real'].max(), df_resultado['Previsto'].max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Perfeição')
axes[0, 1].set_xlabel('Real', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Previsto', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Dispersão: Qual é a precisão?', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. Erro ao longo do tempo
axes[1, 0].bar(df_resultado['Data'], df_resultado['Erro'], 
              color='#e67e22', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1, 0].axhline(y=mae_test, color='red', linestyle='--', linewidth=2.5, 
                  label=f'MAE = {mae_test:.0f} casos')
axes[1, 0].set_ylabel('Erro Absoluto', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Evolução do Erro', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Distribuição de erros
axes[1, 1].hist(df_resultado['Erro'], bins=30, color='#16a085', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=mae_test, color='red', linestyle='--', linewidth=2.5, 
                  label=f'Média: {mae_test:.0f}')
axes[1, 1].set_xlabel('Erro (casos)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequência', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Distribuição dos Erros', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f' ATO IV - RESULTADOS: R² = {r2_test:.1%}  |  MAE = {mae_test:.0f} casos/semana', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"\n MÉTRICAS DE DESEMPENHO:")
print(f"   • R² Score: {r2_test:.1%} (explica {r2_test:.1%} da variação)")
print(f"   • MAE: {mae_test:.0f} casos/semana")
print(f"   • RMSE: {rmse_test:.0f} casos/semana")
print(f"\n CONCLUSÃO: O modelo FUNCIONA! Consegue prever com ~{mae_test:.0f} casos de erro.")

---

# ATO V: A APLICAÇÃO - RECOMENDAÇÕES PRÁTICAS

*Agora que sabemos que funciona, como aplicar isso na prática?*

## 9. Gráfico 6️ : RESUMO EXECUTIVO

**O que conta a história:** Um painel visual resumindo todo o projeto em um só gráfico.

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

# 1. Box com números principais
ax_info = fig.add_subplot(gs[0, :])
ax_info.axis('off')

info_text = f"""
 RESUMO EXECUTIVO - PREVISÃO DE DENGUE EM OURINHOS

 DADOS                           MODELO                       RESULTADOS                     USO PRÁTICO
• 2014-2026 (12 anos)             • Random Forest                • R²: {r2_test:.1%}                    • Alerta 3-4 semanas
• {len(df_clean)} semanas                    • 500 árvores                  • MAE: {mae_test:.0f} casos/sem      • Auxiliar gestão
• 10 variáveis                     • RobustScaler                • Top feature: temp_lag_4         • Não substitui vigilância
"""

ax_info.text(0.05, 0.5, info_text, fontsize=11, verticalalignment='center',
            fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.2))

# 2. Importância (top 5)
ax1 = fig.add_subplot(gs[1, 0])
top_5 = importancia.sort_values(ascending=False).head(5)
colores_top = plt.cm.Reds(np.linspace(0.4, 0.9, 5))
ax1.barh(range(5), top_5.values, color=colores_top[::-1], edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(5))
ax1.set_yticklabels(top_5.index)
ax1.set_xlabel('Importância (%)', fontsize=10, fontweight='bold')
ax1.set_title('Top 5 Features', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# 3. Performance (métricas)
ax2 = fig.add_subplot(gs[1, 1])
ax2.axis('off')
metricas_text = f"""
 DESEMPENHO

R² Score:  {r2_test:.1%}
  └─ Explica {r2_test:.1%} 
     da variação

MAE:  {mae_test:.0f} casos/semana
  └─ Erro típico em
     anos normais

Status:  EXCELENTE
"""
ax2.text(0.1, 0.5, metricas_text, fontsize=10.5, verticalalignment='center',
        fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.2))

# 4. Recomendações
ax3 = fig.add_subplot(gs[2, :])
ax3.axis('off')

rec_text = f"""
 RECOMENDAÇÕES DE USO

 USE PARA:                                            CUIDADO COM:
  • Alerta precoce (3-4 semanas)                       • Surtos sem precedentes (2025 foi maior que histórico)
  • Auxiliar decisões de saúde pública                 • Ausência de atualização com novos dados
  • Dimensionamento de recursos                        • Falta de dados de cobertura vacinal
  • Identificar períodos de risco                      • Não captura novos sorovares

 PRÓXIMOS PASSOS:
   1. Atualizar modelo com dados de 2026+
   2. Integrar cobertura de Dengvaxia
   3. Adicionar dados de circulação viral
   4. Criar dashboard para gestores
"""

ax3.text(0.05, 0.5, rec_text, fontsize=10, verticalalignment='center',
        fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.3))

plt.suptitle(' ATO V - A APLICAÇÃO: Recomendações Práticas', 
             fontsize=14, fontweight='bold', y=0.98)
plt.show()

print("\n" + "="*70)
print(" FIM DA APRESENTAÇÃO".center(70))
print("="*70)

---

## CONCLUSÃO GERAL

###  A História Completa:

1. **Ato I (O Problema):** Ourinhos sofre com surtos recorrentes de dengue, com picos em anos específicos.

2. **Ato II (O Cenário):** O clima segue um padrão sazonal claro, com verão quente/úmido (ideal para mosquitos) e inverno frio/seco.

3. **Ato III (A Conexão):** A biologia revela que temperatura de hoje afeta casos em 3-4 semanas, permitindo alerta precoce.

4. **Ato IV (A Solução):** Random Forest consegue prever com R²=73% e MAE=~65 casos/semana.

5. **Ato V (A Aplicação):** Modelo é prático para auxiliar gestão, mas precisa de atualização contínua.

###  Insight Final:
**"O clima prepara o terreno, mas é a memória epidemiológica que decide o surto."**

Nem clima sozinho, nem dados de casos passados isoladamente explicam tudo. É a **combinação** que funciona.
